# 04 — Explicaciones LLM para predicciones de escalación

Dado un ticket con score de riesgo y factores SHAP, genera narrativas en español usando Claude (con fallback a flan-t5-small).

In [1]:
import os

import pandas as pd

from itops.config import RAW_TICKETS_CSV
from itops.data.loader import load_tickets
from itops.llm.narrative import NarrativeGenerator
from itops.models.escalation import EscalationModel
from itops.models.explainer import ShapExplainer

df = load_tickets(RAW_TICKETS_CSV)
print(f'Tickets cargados: {len(df):,}')

Tickets cargados: 50,000


## 1. Entrenar modelo y extraer top-10 tickets de mayor riesgo

In [2]:
model = EscalationModel(seed=42)
model.fit(df)
proba = model.predict_proba(df)

top10_idx = proba.argsort()[-10:][::-1]
df_top10 = df.iloc[top10_idx].copy()
df_top10['risk_score'] = proba[top10_idx]
print(f'AUC-ROC: {model.eval_metrics_["auc_roc"]:.3f}')
df_top10[['ticket_id', 'category', 'customer_tier', 'risk_score']].head(10)

AUC-ROC: 0.863


,ticket_id,category,customer_tier,risk_score
611,INC0000612,software,enterprise,0.999820
29950,INC0029951,hardware,enterprise,0.999765
11673,INC0011674,software,premium,0.999746
15376,INC0015377,hardware,basic,0.999743
38598,INC0038599,software,premium,0.999706
36933,INC0036934,software,premium,0.999697
14363,INC0014364,hardware,enterprise,0.999690
24299,INC0024300,software,premium,0.999608
30480,INC0030481,software,standard,0.999600
22782,INC0022783,software,enterprise,0.999576


## 2. SHAP features por ticket

In [3]:
explainer = ShapExplainer(model)
top_shap = explainer.top_features(df_top10, n=3)
top_shap.head()

/Users/julio/Desktop/IT Operations Intelligence Platform/.venv/lib/python3.11/site-packages/shap/explainers/_tree.py:620: UserWarning: LightGBM binary classifier with TreeExplainer shap values output has changed to a list of ndarray
  warnings.warn(


,feature_1,shap_1,feature_2,shap_2,feature_3,shap_3
611,num_comments,4.392032,num_reassignments,3.777688,has_critical_keyword,1.340426
29950,num_comments,4.219865,num_reassignments,3.323302,has_critical_keyword,1.426747
11673,num_reassignments,3.952338,num_comments,3.509183,has_critical_keyword,1.383934
15376,num_comments,4.406944,num_reassignments,3.824302,has_critical_keyword,1.654426
38598,num_reassignments,4.057671,num_comments,3.178169,has_critical_keyword,1.260124


## 3. Generar narrativas LLM

In [4]:
api_key = os.getenv('ANTHROPIC_API_KEY')
gen = NarrativeGenerator(api_key=api_key)

narratives = []
for i, (_, ticket) in enumerate(df_top10.iterrows()):
    shap_row = top_shap.iloc[i]
    top_features = [
        {'feature': shap_row[f'feature_{j}'], 'shap': shap_row[f'shap_{j}']}
        for j in range(1, 4)
    ]
    ticket_ctx = {
        'ticket_id': ticket['ticket_id'],
        'category': ticket['category'],
        'priority': ticket['priority_initial'],
        'customer_tier': ticket['customer_tier'],
        'risk_score': float(ticket['risk_score']),
        'description_snippet': str(ticket['description'])[:200],
    }
    narrative = gen.generate(ticket_ctx, top_features)
    narratives.append({
        'ticket_id': ticket['ticket_id'],
        'risk_score': round(float(ticket['risk_score']), 3),
        'provider': narrative.provider,
        'confidence': round(narrative.confidence, 2),
        'summary': narrative.summary,
        'recommendation': narrative.recommendation,
    })

df_narratives = pd.DataFrame(narratives)
df_narratives[['ticket_id', 'risk_score', 'provider', 'confidence']]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.54k [00:00<?, ?B/s]

...

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


,ticket_id,risk_score,provider,confidence
0,INC0000612,1.0,hf,0.0
1,INC0029951,1.0,hf,0.0
2,INC0011674,1.0,hf,0.0
3,INC0015377,1.0,hf,0.0
4,INC0038599,1.0,hf,0.0
5,INC0036934,1.0,hf,0.0
6,INC0014364,1.0,hf,0.0
7,INC0024300,1.0,hf,0.0
8,INC0030481,1.0,hf,0.0
9,INC0022783,1.0,hf,0.0


## 4. Tabla completa

In [5]:
df_narratives[['ticket_id', 'summary', 'recommendation']].to_string(index=False)

' ticket_id                                   summary                                          recommendation\nINC0000612 No se pudo generar un resumen automático. Revisar el ticket manualmente con el equipo de soporte.\nINC0029951 No se pudo generar un resumen automático. Revisar el ticket manualmente con el equipo de soporte.\nINC0011674 No se pudo generar un resumen automático. Revisar el ticket manualmente con el equipo de soporte.\nINC0015377 No se pudo generar un resumen automático. Revisar el ticket manualmente con el equipo de soporte.\nINC0038599 No se pudo generar un resumen automático. Revisar el ticket manualmente con el equipo de soporte.\nINC0036934 No se pudo generar un resumen automático. Revisar el ticket manualmente con el equipo de soporte.\nINC0014364 No se pudo generar un resumen automático. Revisar el ticket manualmente con el equipo de soporte.\nINC0024300 No se pudo generar un resumen automático. Revisar el ticket manualmente con el equipo de soporte.\nINC003048

## 5. Narrativa completa del ticket de mayor riesgo

In [6]:
best = df_narratives.iloc[0]
print(f'Ticket: {best.ticket_id}')
print(f'Risk score: {best.risk_score}')
print(f'Provider: {best.provider} (confidence: {best.confidence})')
print()
print('RESUMEN:')
print(best.summary)
print()
print('RECOMENDACIÓN:')
print(best.recommendation)

Ticket: INC0000612
Risk score: 1.0
Provider: hf (confidence: 0.0)

RESUMEN:
No se pudo generar un resumen automático.

RECOMENDACIÓN:
Revisar el ticket manualmente con el equipo de soporte.


## Conclusiones

- `NarrativeGenerator` produce explicaciones en español a partir del score de riesgo y los factores SHAP del ticket.
- El campo `provider` indica si la narrativa vino de Claude o del fallback HF.
- El caché SQLite evita llamadas redundantes a la API para tickets similares.
- Las narrativas son consumibles directamente por la API de Fase 5 en el endpoint `POST /explain`.